# 01 — Vision Models: CLIP · Diffusion · GAN

This notebook trains and visualises three generative vision models on MNIST.

```
CLIP      : image ↔ text contrastive matching (InfoNCE loss)
Diffusion : DDPM denoising with EMA + curriculum learning
GAN       : alternating Generator / Discriminator (BCELoss)
```

## Architecture overview

```
  CLIP
  ┌──────────────┐     ┌──────────────┐
  │  CNN image   │     │  Transformer │
  │  encoder     │ ↔ ↔ │  text encoder│
  └──────────────┘     └──────────────┘
         ↑ InfoNCE contrastive loss ↑

  Diffusion (DDPM)
  x₀ ──(forward noise)──▶ xₜ ──(UNet)──▶ ε_pred ── MSE ── ε_true
                  ◀──(reverse sample)──

  GAN
  z ~ N(0,I) ──▶ Generator ──▶ x_fake
                                  │
  x_real ──────────────────▶ Discriminator ──▶ real/fake
```

In [ ]:
# ── Install (Colab only) ──────────────────────────────────────────────────
import sys, os
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_JUPYTER_TOKEN'))
if IN_COLAB:
    !pip install -e . -q

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

from viz import plot_metrics, show_image_grid, show_clip_similarity
from mini_networks.colab.launcher import run_model

---
## 1. CLIP — Contrastive Image-Language Pretraining

**Key idea:** pull together image and text embeddings for matching pairs,
push apart non-matching ones.  Loss = InfoNCE (symmetric cross-entropy over the similarity matrix).

> **Formula:** $L = -\frac{1}{N}\sum_i \log\frac{\exp(s_{ii}/\tau)}{\sum_j \exp(s_{ij}/\tau)}$

In [ ]:
clip_logger = run_model('clip', fast_demo=True)

In [ ]:
plot_metrics(clip_logger, keys=['loss'])

In [ ]:
# ── CLIP inference: visualise cosine similarity matrix ───────────────────
from mini_networks.models.clip.config import CLIPConfig
from mini_networks.models.clip.trainer import CLIPTrainer, make_clip_dataloader
import torch

config = CLIPConfig(fast_demo=True)
trainer = CLIPTrainer()
trainer.load_checkpoint(config, clip_logger.artifacts_dir)

dl = make_clip_dataloader(config, split='train')
images, tokens, labels = next(iter(dl))
images = images[:8].to(config.device)
tokens = tokens[:8].to(config.device)

trainer.model.eval()
with torch.no_grad():
    img_emb = trainer.model.encode_image(images).cpu()
    txt_emb = trainer.model.encode_text(tokens).cpu()

label_names = [str(l.item()) for l in labels[:8]]
show_clip_similarity(img_emb, txt_emb, label_names)

---
## 2. Diffusion — DDPM with EMA

**Key idea:** gradually corrupt images with Gaussian noise (forward process),
train a UNet to predict the noise, then reverse the process at inference.

> **EMA:** exponential moving average of weights → smoother samples at inference.  
> **Curriculum:** sort batches by pixel variance so the model sees hard examples first.

In [ ]:
diff_logger = run_model('diffusion', fast_demo=True)

In [ ]:
plot_metrics(diff_logger, keys=['loss'])

In [ ]:
# ── Generate samples ─────────────────────────────────────────────────────
from mini_networks.models.diffusion.config import DiffusionConfig
from mini_networks.models.diffusion.trainer import DDPMTrainer

config = DiffusionConfig(fast_demo=True)
trainer = DDPMTrainer()
trainer.load_checkpoint(config, diff_logger.artifacts_dir)

result = trainer.infer(config, {'n_samples': 8})
show_image_grid(result['samples'], title='Diffusion samples (DDPM)', nrow=4)

---
## 3. GAN — Generative Adversarial Network

**Key idea:** a Generator and a Discriminator play a min-max game.

```
G: z → x_fake    (fool D)
D: x → [0,1]     (distinguish real from fake)
```

> **Loss:** $\min_G \max_D \;\mathbb{E}[\log D(x)] + \mathbb{E}[\log(1-D(G(z)))]$

In [ ]:
gan_logger = run_model('gan', fast_demo=True)

In [ ]:
plot_metrics(gan_logger, keys=['d_loss', 'g_loss'])

In [ ]:
# ── Generate GAN samples ─────────────────────────────────────────────────
from mini_networks.models.gan.config import GANConfig
from mini_networks.models.gan.trainer import GANTrainer

config = GANConfig(fast_demo=True)
trainer = GANTrainer()
trainer.load_checkpoint(config, gan_logger.artifacts_dir)

result = trainer.infer(config, {'n_samples': 8, 'seed': 42})
show_image_grid(result['samples'], title='GAN samples', nrow=4)

---
## 4. Checkpoint roundtrip demo

Demonstrates that `load_checkpoint()` + `infer()` reproduces the same result.

In [ ]:
import torch

# Reload GAN from checkpoint and generate identical samples (same seed)
fresh_trainer = GANTrainer()
fresh_trainer.load_checkpoint(GANConfig(fast_demo=True), gan_logger.artifacts_dir)
result2 = fresh_trainer.infer(GANConfig(fast_demo=True), {'n_samples': 8, 'seed': 42})

match = torch.allclose(result['samples'], result2['samples'])
print(f'Checkpoint roundtrip identical: {match}')
show_image_grid(result2['samples'], title='GAN samples (reloaded checkpoint)', nrow=4)